# Faster R-CNN Experiments: GOST Stamp Detection

**Цель:** Обучить Faster R-CNN для детекции штампов на строительных чертежах.

**Данные:** 500 synthetic train_v3 + 10 real val_honest + 35 real test

**Метрики:** IoU, Precision, Recall, F1 на 49 реальных изображениях

**Подход:** Single training run, ResNet50 FPN backbone, 30 epochs, GPU T4 (Colab)

**Resize:** Компромисс — synthetic (200 DPI) ресайзится по PPI (100), real — fallback на 800px.


In [1]:
import random
import sys
from pathlib import Path
import yaml

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

with open(PROJECT_DIR / "configs" / "config.yaml") as f:
    cfg = yaml.safe_load(f)


## 2. Imports & Data Setup

In [2]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torch.utils.data import DataLoader, Subset

from src.evaluation import set_seeds
from src.evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from src.data.loader import load_image_and_labels
from src.config import test_image_dir, test_label_dir, get_path, load_donors, load_val_honest

RANDOM_STATE = cfg["synthetic"]["random_state"]
set_seeds(RANDOM_STATE)
print(f"Seed loaded from config: RANDOM_STATE = {RANDOM_STATE} (synthetic.random_state)")

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = test_image_dir(cfg)
LABEL_TEST_DIR = test_label_dir(cfg)
IMAGE_TRAIN_DIR = get_path(cfg, "images_train")
LABEL_TRAIN_DIR = get_path(cfg, "labels_train")

DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")
print(f"Train images: {len(list(IMAGE_TRAIN_DIR.glob('*.png')) + list(IMAGE_TRAIN_DIR.glob('*.jpg')))}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png')) + list(IMAGE_TEST_DIR.glob('*.jpg')))}")

DONORS = load_donors(cfg)
VAL_HONEST = load_val_honest(cfg)
TEST_EXCLUDE = DONORS | VAL_HONEST
print(f"Donors excluded from eval: {sorted(DONORS)}")
print(f"Val_honest excluded from eval: {sorted(VAL_HONEST)}")
print(f"Test images will be: 49 - {len(TEST_EXCLUDE)} = {49 - len(TEST_EXCLUDE)}")

Seed loaded from config: RANDOM_STATE = 42 (synthetic.random_state)
Device: cpu
Train images: 500
Test images: 49
Donors excluded from eval: ['test_11.png', 'test_17.png', 'test_23.png', 'test_42.png']
Val_honest excluded from eval: ['test_04.png', 'test_06.png', 'test_07.png', 'test_08.png', 'test_29.jpg', 'test_35.jpg', 'test_37.jpg', 'test_38.jpg', 'test_44.png', 'test_46.png']
Test images will be: 49 - 14 = 35


## 3. Dataset & DataLoader

**Компромисс PPI:**
- Synthetic (train): DPI=200 известен → ресайз по целевому PPI (100).
  Штамп 185×55mm всегда ~728×216px — единый физический масштаб.
- Real (test): PPI неизвестен → fallback на MAX_SIZE=800 по длинной стороне.
- Bbox корректируются пропорционально scale.
- Цель: synthetic учится на физических размерах, real не ломается.

In [3]:
from src.data.loader import GOSTDataset, TARGET_PPI, SYNTH_DPI, MAX_SIZE

train_dataset = GOSTDataset(IMAGE_TRAIN_DIR, LABEL_TRAIN_DIR, synth_dpi=SYNTH_DPI)
test_dataset = GOSTDataset(IMAGE_TEST_DIR, LABEL_TEST_DIR)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

Train: 500, Test: 49


## 4. Data Split (v4 baseline)

Synthetic (500) → train (v3); Real (49) → test.
4 donors + 10 val_honest excluded from eval metrics.
Eval on 35 non-donor, non-val images gives unbiased accuracy estimate.

In [4]:
# ─── Synthetic 80/20 split (400 train + 100 val)
n_synth = len(train_dataset)
synth_indices = list(range(n_synth))
random.Random(RANDOM_STATE).shuffle(synth_indices)
n_val_synth = int(0.2 * n_synth)
val_synth_idx = synth_indices[:n_val_synth]
train_synth_idx = synth_indices[n_val_synth:]
print(f"Synthetic: {len(train_synth_idx)} train + {len(val_synth_idx)} val")

# ─── Real 80/20 split (39 val + 10 holdout test)
all_real_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
random.Random(RANDOM_STATE).shuffle(all_real_images)
n_val_real = int(0.8 * len(all_real_images))
val_images = all_real_images[:n_val_real]
test_images = all_real_images[n_val_real:]
print(f"Real: {len(val_images)} val + {len(test_images)} holdout test")
for p in test_images:
    print(f"  Holdout: {p.name}")

Synthetic: 400 train + 100 val
Real: 39 val + 10 holdout test
  Holdout: test_06.png
  Holdout: test_22.jpg
  Holdout: test_07.png
  Holdout: test_10.png
  Holdout: test_16.png
  Holdout: test_17.png
  Holdout: test_21.png
  Holdout: test_02.png
  Holdout: test_08.png
  Holdout: test_33.jpg


## 5. Training

Оптимизатор: Adam с дифференциальным LR (backbone `1e-4`, head `1e-3`).
Synthetic 500 all train (random 80/20 split for early stopping val). NaN/Inf check.

In [ ]:
import time
start = time.time()

NUM_CLASSES = 2
NUM_EPOCHS = cfg["rcnn"]["num_epochs"]
BATCH_SIZE = cfg["rcnn"]["batch_size"]
ACCUM_STEPS = cfg["rcnn"]["accum_steps"]
LR_HEAD = cfg["rcnn"]["learning_rate"]
LR_BACKBONE = cfg["rcnn"]["lr_backbone"]

from src.models.rcnn_model import RCNNModel
rcnn = RCNNModel(num_classes=NUM_CLASSES, device=DEVICE)
rcnn.build(weights="DEFAULT")
model = rcnn.model

# Differential LR
backbone_params = []
head_params = []
for name, p in model.named_parameters():
    if 'box_predictor' in name:
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.Adam([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

train_subset = Subset(train_dataset, train_synth_idx)
val_subset = Subset(train_dataset, val_synth_idx)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, collate_fn=lambda x: tuple(zip(*x)), num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, collate_fn=lambda x: tuple(zip(*x)), num_workers=0, pin_memory=True)

best_val_loss = float('inf')
patience = 5
wait = 0
best_epoch = 0
train_loss_history = []
val_loss_history = []

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    for batch_idx, (images, targets) in enumerate(train_loader):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        if not torch.isfinite(losses):
            print(f"NaN/Inf at epoch {epoch+1}, batch {batch_idx}, skipping")
            continue

        losses = losses / ACCUM_STEPS
        losses.backward()

        if (batch_idx + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += losses.item() * ACCUM_STEPS
    lr_scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)

    # Validate (train mode + no_grad for proper loss computation)
    model.train()
    with torch.no_grad():
        val_loss = 0
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            output = model(images, targets)

            if isinstance(output, dict):
                val_loss += sum(v.item() for v in output.values())
            elif isinstance(output, list):
                for d in output:
                    if isinstance(d, dict):
                        val_loss += sum(v.item() for v in d.values())
    avg_val_loss = val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        wait = 0
        torch.save(model.state_dict(), str(ARTIFACTS_DIR / "rcnn" / "v4" / "rcnn_best.pth"))
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}, best epoch {best_epoch+1}, best val loss {best_val_loss:.4f}")
            break

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

print(f"Weights saved to {ARTIFACTS_DIR / 'rcnn' / 'v4' / 'rcnn_best.pth'}")
loss_history = train_loss_history
val_loss_history_print = val_loss_history

## 6. Loss Plot

Train + Val loss кривые. Early stopping отсекает лишние эпохи.

In [ ]:
plt.figure(figsize=(10, 5))
best_epoch_display = best_epoch + 1
plt.plot(range(1, len(train_loss_history)+1), train_loss_history, marker="o", label="Train Loss")
plt.plot(range(1, len(val_loss_history)+1), val_loss_history, marker="s", label="Val Loss")
plt.axvline(x=best_epoch_display, color="r", linestyle="--", alpha=0.5, label=f"Best epoch {best_epoch_display}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_loss.png", dpi=150)
plt.show()

## 7. Evaluation on 35 Non-Donor, Non-Val Images

All 49 real images = test set; 4 donors + 10 val_honest excluded from metrics.
Threshold sweep `[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5]` + greedy matching (max IoU per GT).

In [ ]:
from src.evaluation.evaluate_rcnn import evaluate_rcnn

# Use module-level evaluate_rcnn() with threshold sweep + greedy matching
metrics, results_list = evaluate_rcnn(
    model, IMAGE_TEST_DIR, LABEL_TEST_DIR, TEST_EXCLUDE,
    conf_thresholds=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5],
)
print_metrics(metrics, prefix="RCNN ")


## 8. Visualization

Worst/Best по IoU среди 10 holdout.

In [ ]:
sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in all_real_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (Faster R-CNN)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_best_worst.png", dpi=150)
plt.show()

## 9. Conclusions

Итог: Faster R-CNN на 35 non-donor (4 donors + 10 val_honest excluded). Differential LR, Adam, early stopping.

In [ ]:
summary = {
    "model": "Faster R-CNN (ResNet50 FPN)",
    "optimizer": "Adam",
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD,
    "data": "500 synthetic v4 (50 GOST + 250 copy-paste + 200 GOST-on-real-bg) + 49 real test (eval on 35 non-donor, non-val)",
    "best_epoch": best_epoch_display,
    "epochs": len(train_loss_history),
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "rcnn_v4_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/rcnn_v4_results.json")
print(json.dumps(summary, indent=2))